# GrumpReLU Implementation

JumpReLU is a really useful activation function, but it is tailored for single dimensional features. This notebook implements and tests a variant of JumpReLU for *groups* of latent variables.

In [51]:
import torch
import numpy as np
from typing import Any
import einops as eo

The first thing we need to implement is an approximation for the Dirac delta function using the Rectangle function. JumpReLU uses the Heaviside step function, who's derivative is the Dirac delta, which is entirely unworkable for SGD methods. 

In [52]:
def rectangle_bandwidth(x : torch.Tensor, bandwidth : float) -> torch.Tensor:
    rectangle = (-bandwidth/2 < x) & (x < bandwidth / 2)
    
    return rectangle / bandwidth

Now, we need to implement the GrumpReLU function in the glorious, yet horrendous, manner required by Torch.

In [53]:
def grump_relu_forward(x : torch.Tensor, threshold : torch.Tensor):
    """
    GrumpReLU forward implementation. 
    
    Defined as a separate function since it will be used both in the forward pass of the Autograd function as well as in the GrumpReLU layer.
    """

    norms = x.norm(dim=-1)
    mask = norms > threshold
    
    return x * mask.unsqueeze(-1) 

class GrumpReLU(torch.autograd.Function):
    @staticmethod
    def forward(
        x : torch.Tensor, 
        threshold : torch.Tensor, 
        bandwidth : float
        ) -> torch.Tensor:

        # x, output : (..., n_experts, d_expert)
        return grump_relu_forward(x, threshold)

    @staticmethod
    def setup_context(
        ctx : Any, 
        inputs: tuple[torch.Tensor, torch.Tensor, float], 
        output: torch.Tensor
        ) -> None:
        x, threshold, bandwidth = inputs
        del output

        ctx.save_for_backward(x, threshold)
        ctx.bandwidth = bandwidth

    @staticmethod
    def backward(
        ctx : Any, 
        grad_outputs : torch.Tensor
        ) -> tuple[torch.Tensor, torch.Tensor, None]:

        x, threshold = ctx.saved_tensors
        bandwidth = ctx.bandwidth

        # x, grad_output : ..., n_experts, d_expert
        # threshold: n_experts
        # Want: grad sizes to be ..., n_experts, d_expert

        norms = x.norm(dim=-1) # ..., n_experts
        mask = (norms > threshold).to(x)

        # All dimensions of the output have derivative 1 wrt the x
        x_grad = eo.repeat(mask, '... n_experts -> ... n_experts d_expert', d_expert=x.shape[-1]) * grad_outputs

        # x_grad = eo.reduce(partials * grad_outputs, '... n_experts d_expert -> n_experts d_expert', reduction='sum')

        # Threshold is the same for every sample in the batch (BROADCAST OVER THE BATCH) so requires a sum
        threshold_grad = eo.reduce(
             - x * 
             (threshold * rectangle_bandwidth(norms - threshold, ctx.bandwidth) / norms).unsqueeze(-1) 
             * grad_outputs,
             '... n_experts d_expert -> n_experts',
             reduction='sum'
        )

        return (x_grad, threshold_grad, None)


## Learning test harness

Sanity check that the threshold is the *only* thing that learns, and that it learns to keep a signal manifold while filtering noise.

`run_threshold_test` is a reusable base: pass a `sample_signal` function (manifold to keep) and an optional `sample_noise` function (points to zero out), and it trains `theta` end to end.

- **PGD on theta:** `theta` is trained *directly* (no $e^t$ reparam). Each step is a plain Adam update followed by a projection back onto $\{\theta \ge 0\}$ (`theta.clamp_(min=0)`). This reaches small $\theta$ far faster than the exp trick and lets it hit exactly 0.
- **Train / hold-out split:** fresh samples for each, so accuracy is on data the threshold never saw.
- **Mini-batching:** `batch_size` (`None` = full batch) — stochastic batches inject the gradient noise that helps θ escape the flat region above the signal norm.
- **WSD schedule:** warmup → stable → linear decay to 0; the decay phase shrinks the step below the keep-gap width so θ settles *inside* it.
- **Optional `l1`:** a small L1 pull on θ. The residual floor in the no-noise case is geometric (near θ=0 the misclassified set has measure ~θ and its gradient vanishes), *not* a parametrization artifact — a tiny constant downward force plus the projection snaps θ to exactly 0, while leaving the gap cases untouched.
- **Loss:** keep/zero classification — target is `x` for signal, `0` for noise (MSE on the output vectors).

Since GrumpReLU gates purely on the group **norm**, signal manifolds sit on the unit shell ($\lVert x\rVert = 1$, the shape sets only the angular pattern) and noise **hugs near the signal points** but pulled just inside, so a single scalar $\theta$ must find the gap.

In [54]:
def wsd_lr(step, total_steps, base_lr, warmup_frac=0.05, decay_frac=0.25):
    """Warmup-Stable-Decay schedule: linear warmup, constant plateau, then linear decay to 0.

    The decay phase shrinks the step below the keep-gap width so theta settles *inside*
    the gap instead of striding over it (Adam's RMS-normalised step is ~constant otherwise).
    """
    warmup = max(1, int(total_steps * warmup_frac))
    decay = max(1, int(total_steps * decay_frac))
    decay_start = total_steps - decay
    if step < warmup:
        return base_lr * (step + 1) / warmup
    if step < decay_start:
        return base_lr
    return base_lr * max(0.0, (total_steps - step) / decay)


def run_threshold_test(
    sample_signal,                 # (n, d) -> signal points to KEEP
    sample_noise=None,             # (n, d) -> noise points to ZERO OUT (None = no noise)
    *,
    name="",
    d_expert=3,
    n_train=2048,
    n_holdout=4096,
    batch_size=256,                # None -> full batch
    steps=600,
    lr=0.05,
    bandwidth=0.5,
    init_theta=0.5,
    l1=0.0,                        # optional L1 pull on theta -> drives it to exactly 0 when nothing opposes
    warmup_frac=0.05,
    decay_frac=0.25,
    seed=0,
    verbose=False,
):
    """Reusable GrumpReLU threshold test: theta is the only trainable parameter.

    theta is trained directly (no exp reparam) by projected gradient descent: a plain Adam
    step followed by a projection back onto {theta >= 0}. Trains on a mini-batched train
    split with a WSD lr schedule and reports accuracy on a fresh hold-out split.
    """
    torch.manual_seed(seed)

    def make_split(n):
        signal = sample_signal(n, d_expert)
        noise = sample_noise(n, d_expert) if sample_noise is not None else signal.new_zeros((0, d_expert))
        x = torch.cat([signal, noise], dim=0).unsqueeze(1)            # (N, n_experts=1, d_expert)
        target = torch.cat([signal, torch.zeros_like(noise)], dim=0).unsqueeze(1)
        labels = torch.cat([torch.ones(len(signal)), torch.zeros(len(noise))])  # 1 == keep
        return x, target, labels, signal.norm(dim=-1)

    x_tr, tgt_tr, lab_tr, sig_norm = make_split(n_train)
    x_ho, _, lab_ho, _ = make_split(n_holdout)

    N = x_tr.shape[0]
    bs = N if (batch_size is None or batch_size >= N) else batch_size

    def accuracy(x, labels, theta):
        with torch.no_grad():
            out = GrumpReLU.apply(x, theta, bandwidth)
            return ((out.norm(dim=-1).squeeze(-1) > 0).float() == labels).float().mean().item()

    # theta is the raw trainable parameter; PGD projects it back onto theta >= 0 each step
    theta = torch.nn.Parameter(torch.tensor([float(init_theta)]))
    opt = torch.optim.Adam([theta], lr=lr)

    for step in range(steps):
        opt.param_groups[0]["lr"] = wsd_lr(step, steps, lr, warmup_frac, decay_frac)
        idx = torch.arange(N) if bs >= N else torch.randint(0, N, (bs,))
        opt.zero_grad()
        out = GrumpReLU.apply(x_tr[idx], theta, bandwidth)
        loss = ((out - tgt_tr[idx]) ** 2).sum(dim=-1).mean() + l1 * theta.abs().sum()
        loss.backward()
        opt.step()
        with torch.no_grad():
            theta.clamp_(min=0.0)  # projection onto the feasible set {theta >= 0}

        if verbose and (step % max(1, steps // 6) == 0 or step == steps - 1):
            print(f"  step {step:5d}  lr {opt.param_groups[0]['lr']:.4f}  theta {theta.item():.3f}  "
                  f"train_acc {accuracy(x_tr, lab_tr, theta):.3f}")

    train_acc = accuracy(x_tr, lab_tr, theta)
    holdout_acc = accuracy(x_ho, lab_ho, theta)
    print(f"[{name:14s}] theta={theta.item():.3f}  train_acc={train_acc:.3f}  holdout_acc={holdout_acc:.3f}  "
          f"signal-norm in [{sig_norm.min():.3f}, {sig_norm.max():.3f}]")
    return dict(name=name, theta=theta.item(), train_acc=train_acc, holdout_acc=holdout_acc)

In [55]:
import math


# --- signal manifolds (all placed on the unit shell) ---
def sphere(n, d):
    """Uniform points on the unit sphere."""
    v = torch.randn(n, d)
    return v / v.norm(dim=-1, keepdim=True)


def torus(n, d, R=2.0, r=0.7):
    """Torus surface, projected onto the unit shell (angular pattern only)."""
    assert d == 3, "torus is defined in 3D"
    u = 2 * math.pi * torch.rand(n)
    v = 2 * math.pi * torch.rand(n)
    pts = torch.stack([(R + r * v.cos()) * u.cos(),
                       (R + r * v.cos()) * u.sin(),
                       r * v.sin()], dim=-1)
    return pts / pts.norm(dim=-1, keepdim=True)


def spiral(n, d, turns=5):
    """Spherical spiral -- already lies on the unit sphere."""
    assert d == 3, "spiral is defined in 3D"
    s = torch.rand(n)
    z = 2 * s - 1
    rad = (1 - z ** 2).clamp_min(0).sqrt()
    ang = 2 * math.pi * turns * s
    return torch.stack([rad * ang.cos(), rad * ang.sin(), z], dim=-1)


def hugging_noise(sample_signal, lo=0.3, hi=0.98):
    """Noise that hugs near the signal points (same directions) but pulled just inside the shell."""
    def sampler(n, d):
        pts = sample_signal(n, d)
        scale = lo + (hi - lo) * torch.rand(pts.shape[0], 1)
        return pts * scale
    return sampler

In [56]:
# Tight gap (noise up to 0.98 of the shell): batched + WSD lets theta settle in (0.98, 1.0).
_ = run_threshold_test(sphere, hugging_noise(sphere), name="sphere")
_ = run_threshold_test(torus,  hugging_noise(torus),  name="torus")
_ = run_threshold_test(spiral, hugging_noise(spiral), name="spiral", verbose=True)

[sphere        ] theta=0.991  train_acc=1.000  holdout_acc=1.000  signal-norm in [1.000, 1.000]
[torus         ] theta=0.984  train_acc=1.000  holdout_acc=1.000  signal-norm in [1.000, 1.000]
  step     0  lr 0.0017  theta 0.502  train_acc 0.643
  step   100  lr 0.0500  theta 0.979  train_acc 0.999
  step   200  lr 0.0500  theta 0.942  train_acc 0.973
  step   300  lr 0.0500  theta 0.914  train_acc 0.952
  step   400  lr 0.0500  theta 0.960  train_acc 0.985
  step   500  lr 0.0333  theta 0.938  train_acc 0.970
  step   599  lr 0.0003  theta 0.995  train_acc 1.000
[spiral        ] theta=0.995  train_acc=1.000  holdout_acc=1.000  signal-norm in [1.000, 1.000]


## Degenerate case: threshold must collapse to zero

A spiral that runs **through the origin** with **no noise**: every point must pass, so the correct threshold is $\theta \to 0$.

With PGD, $\theta$ descends quickly, but stalls just above 0 on its own: near $\theta=0$ only a measure-$\sim\theta$ sliver of near-origin points is still misclassified and their gradient vanishes ($\propto n^2$), so there is nothing left to push $\theta$ down. A tiny `l1` adds a constant downward force; combined with the $\{\theta \ge 0\}$ projection it pins $\theta$ to **exactly 0**.

In [71]:
def origin_spiral(n, d, a=1.5, turns=3):
    """Planar spiral running from ~0 outward -> norms span [~0, a], passing through the origin."""
    assert d == 3, "origin spiral is defined in 3D"
    s = torch.rand(n).clamp_min(1e-3)  # near 0, but not exactly (avoids 0/0 in the backward)
    r = a * s
    ang = 2 * math.pi * turns * s
    return torch.stack([r * ang.cos(), r * ang.sin(), torch.zeros_like(s)], dim=-1)


# No noise -> theta must collapse to 0. A tiny L1 + the PGD projection pins it to exactly 0.
_ = run_threshold_test(origin_spiral, None, name="origin-spiral",
                       init_theta=0.9, lr=0.05, steps=15000, l1=0, verbose=True)

  step     0  lr 0.0001  theta 0.900  train_acc 0.398
  step  2500  lr 0.0500  theta 0.015  train_acc 0.989
  step  5000  lr 0.0500  theta 0.006  train_acc 0.995
  step  7500  lr 0.0500  theta 0.003  train_acc 0.998
  step 10000  lr 0.0500  theta 0.001  train_acc 1.000
  step 12500  lr 0.0333  theta 0.001  train_acc 1.000
  step 14999  lr 0.0000  theta 0.001  train_acc 1.000
[origin-spiral ] theta=0.001  train_acc=1.000  holdout_acc=1.000  signal-norm in [0.002, 1.500]
